## Text Classification With RNN & LSTM
### Name - Tushar Dugane
### PRN - SCFP125012

Activity 1: Text Classification using RNN

IMDB Movie Reviews Dataset

In [ ]:
# -----------------------------
# 1. Import Libraries
# -----------------------------

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import tiktoken

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# -----------------------------
# 2. Load Dataset
# -----------------------------

df = pd.read_csv("IMDB Dataset.csv")

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (50000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [ ]:
# -----------------------------
# 3. Take a Smaller Balanced Dataset
# -----------------------------

positive = df[df["sentiment"] == "positive"].sample(
    n=1000, random_state=42
)

negative = df[df["sentiment"] == "negative"].sample(
    n=1000, random_state=42
)

df_small = pd.concat([positive, negative])

# Shuffle the dataset
df_small = df_small.sample(frac=1, random_state=42).reset_index(drop=True)

print("Small dataset shape:", df_small.shape)
print(df_small["sentiment"].value_counts())

Small dataset shape: (2000, 2)
sentiment
negative    1000
positive    1000
Name: count, dtype: int64


In [ ]:
# -----------------------------
# 4. Convert Labels
# -----------------------------

df_small["label"] = df_small["sentiment"].map({
    "negative": 0,
    "positive": 1
})

print(df_small[["sentiment", "label"]].head())

  sentiment  label
0  negative      0
1  positive      1
2  negative      0
3  positive      1
4  negative      0


In [ ]:
# -----------------------------
# 5. Train-Test Split
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    df_small["review"].values,
    df_small["label"].values,
    test_size=0.2,
    random_state=42,
    stratify=df_small["label"].values
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 1600
Testing samples: 400


In [ ]:
# -----------------------------
# 6. Load tiktoken
# -----------------------------

encoding = tiktoken.get_encoding("cl100k_base")

print("Tokenizer loaded successfully!")

# Example
sample_text = "This movie was absolutely fantastic!"

tokens = encoding.encode(sample_text)

print("Original text:")
print(sample_text)

print("\nToken IDs:")
print(tokens)

Tokenizer loaded successfully!
Original text:
This movie was absolutely fantastic!

Token IDs:
[2028, 5818, 574, 11112, 14964, 0]


In [ ]:
# -----------------------------
# 7. Tokenize Reviews
# -----------------------------

def tokenize_text(text):
    return encoding.encode(text)

X_train_tokens = [tokenize_text(text) for text in X_train]
X_test_tokens = [tokenize_text(text) for text in X_test]

print("First review:")
print(X_train[0][:200])

print("\nFirst review tokens:")
print(X_train_tokens[0][:20])

First review:
This movies chronicles the life and times of William Castle. He made a series of low budget horror films in the 1950s-1960s that he sold with gimmicks. In "13 Ghosts" you need viewers to see the ghost

First review tokens:
[2028, 9698, 27076, 4440, 279, 2324, 323, 3115, 315, 12656, 27987, 13, 1283, 1903, 264, 4101, 315, 3428, 8199, 22169]


In [ ]:
# -----------------------------
# 8. Padding / Truncation
# -----------------------------

MAX_LENGTH = 200

def pad_sequence(tokens, max_length=MAX_LENGTH):

    # Truncate
    if len(tokens) > max_length:
        tokens = tokens[:max_length]

    # Padding
    else:
        tokens = tokens + [0] * (max_length - len(tokens))

    return tokens


X_train_padded = [
    pad_sequence(tokens)
    for tokens in X_train_tokens
]

X_test_padded = [
    pad_sequence(tokens)
    for tokens in X_test_tokens
]

print("Number of tokens in first review:")
print(len(X_train_padded[0]))

print("Maximum sequence length:", MAX_LENGTH)

Number of tokens in first review:
200
Maximum sequence length: 200


In [ ]:
# -----------------------------
# 9. Convert to PyTorch Tensors
# -----------------------------

X_train_tensor = torch.tensor(
    X_train_padded,
    dtype=torch.long
)

X_test_tensor = torch.tensor(
    X_test_padded,
    dtype=torch.long
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)

print("X_train shape:", X_train_tensor.shape)
print("X_test shape:", X_test_tensor.shape)
print("y_train shape:", y_train_tensor.shape)
print("y_test shape:", y_test_tensor.shape)

X_train shape: torch.Size([1600, 200])
X_test shape: torch.Size([400, 200])
y_train shape: torch.Size([1600])
y_test shape: torch.Size([400])


In [ ]:
# -----------------------------
# 10. Create DataLoaders
# -----------------------------

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("DataLoaders created!")

DataLoaders created!


In [ ]:
# -----------------------------
# 11. Define RNN Model
# -----------------------------

class RNNClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=64,
        hidden_dim=64
    ):

        super().__init__()

        # Embedding layer
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        # RNN layer
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # Output layer
        self.fc = nn.Linear(
            hidden_dim,
            2
        )

    def forward(self, x):

        # Convert token IDs to embeddings
        x = self.embedding(x)

        # Pass through RNN
        output, hidden = self.rnn(x)

        # Take the last time step
        x = output[:, -1, :]

        # Classification
        x = self.fc(x)

        return x

In [ ]:
# -----------------------------
# 12. Vocabulary Size
# -----------------------------

vocab_size = encoding.n_vocab

print("Vocabulary size:", vocab_size)

Vocabulary size: 100277


In [ ]:
# -----------------------------
# 13. Create Model
# -----------------------------

model = RNNClassifier(
    vocab_size=vocab_size,
    embedding_dim=64,
    hidden_dim=64
)

print(model)

RNNClassifier(
  (embedding): Embedding(100277, 64, padding_idx=0)
  (rnn): RNN(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


In [ ]:
# -----------------------------
# 14. Loss and Optimizer
# -----------------------------

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [ ]:
# -----------------------------
# 15. Train the Model
# -----------------------------

epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch_X, batch_y in train_loader:

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(batch_X)

        # Calculate loss
        loss = criterion(outputs, batch_y)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/5], Loss: 0.7058
Epoch [2/5], Loss: 0.6739
Epoch [3/5], Loss: 0.6523
Epoch [4/5], Loss: 0.6195
Epoch [5/5], Loss: 0.5750


In [ ]:
# -----------------------------
# 16. Evaluate Model
# -----------------------------

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch_X, batch_y in test_loader:

        outputs = model(batch_X)

        # Get predicted class
        _, predicted = torch.max(outputs, 1)

        total += batch_y.size(0)

        correct += (
            predicted == batch_y
        ).sum().item()

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 51.25%


In [ ]:
# -----------------------------
# 17. Test New Reviews
# -----------------------------

def predict_sentiment(review):

    # Tokenize
    tokens = encoding.encode(review)

    # Pad / truncate
    tokens = pad_sequence(tokens)

    # Convert to tensor
    tensor = torch.tensor(
        [tokens],
        dtype=torch.long
    )

    # Prediction
    model.eval()

    with torch.no_grad():

        output = model(tensor)

        prediction = torch.argmax(
            output,
            dim=1
        ).item()

    if prediction == 1:
        return "Positive"
    else:
        return "Negative"


# New reviews
reviews = [
    "This movie was absolutely fantastic. I loved every minute of it.",
    "The movie was boring and disappointing. I did not enjoy it.",
    "The acting was excellent and the story was very entertaining."
]

for review in reviews:

    prediction = predict_sentiment(review)

    print("\nReview:", review)
    print("Predicted Sentiment:", prediction)


Review: This movie was absolutely fantastic. I loved every minute of it.
Predicted Sentiment: Negative

Review: The movie was boring and disappointing. I did not enjoy it.
Predicted Sentiment: Negative

Review: The acting was excellent and the story was very entertaining.
Predicted Sentiment: Negative


Activity 2: Text Classification using LSTM

IMDB Movie Reviews Dataset

In [ ]:
!pip install pandas scikit-learn torch tiktoken

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import tiktoken

from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# ============================================
# LOAD DATASET
# ============================================

FILE_PATH = "IMDB Dataset.csv"

try:

    df = pd.read_csv(
        FILE_PATH,
        engine="python",
        on_bad_lines="skip"
    )

except FileNotFoundError:

    print("File not found!")
    print("Make sure IMDB Dataset(1).csv is uploaded to your notebook.")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

Dataset loaded successfully!
Dataset shape: (50000, 2)

Columns:
Index(['review', 'sentiment'], dtype='object')

Sentiment distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [ ]:
# ============================================
# SELECT 2000 BALANCED REVIEWS
# ============================================

positive = df[
    df["sentiment"] == "positive"
].sample(
    n=1000,
    random_state=42
)

negative = df[
    df["sentiment"] == "negative"
].sample(
    n=1000,
    random_state=42
)

df_small = pd.concat([
    positive,
    negative
])

# Shuffle
df_small = df_small.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Small dataset shape:", df_small.shape)

print("\nSentiment distribution:")
print(df_small["sentiment"].value_counts())

Small dataset shape: (2000, 2)

Sentiment distribution:
sentiment
negative    1000
positive    1000
Name: count, dtype: int64


In [ ]:
# ============================================
# CONVERT LABELS
# ============================================

df_small["label"] = df_small["sentiment"].map({
    "negative": 0,
    "positive": 1
})

print(df_small[[
    "sentiment",
    "label"
]].head(10))

  sentiment  label
0  negative      0
1  positive      1
2  negative      0
3  positive      1
4  negative      0
5  negative      0
6  positive      1
7  negative      0
8  positive      1
9  negative      0


In [ ]:
# ============================================
# TRAIN / TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    df_small["review"].values,
    df_small["label"].values,
    test_size=0.2,
    random_state=42,
    stratify=df_small["label"].values
)

print("Training reviews:", len(X_train))
print("Testing reviews :", len(X_test))

Training reviews: 1600
Testing reviews : 400


In [ ]:
# ============================================
# TOKENIZATION
# ============================================

encoding = tiktoken.get_encoding(
    "cl100k_base"
)

print("tiktoken loaded successfully!")
print("Vocabulary size:", encoding.n_vocab)

tiktoken loaded successfully!
Vocabulary size: 100277


In [ ]:
# ============================================
# TOKENIZE REVIEWS
# ============================================

X_train_tokens = [
    encoding.encode(text)
    for text in X_train
]

X_test_tokens = [
    encoding.encode(text)
    for text in X_test
]

print("Tokenization completed!")

print("\nExample review:")
print(X_train[0][:200])

print("\nFirst 20 token IDs:")
print(X_train_tokens[0][:20])

Tokenization completed!

Example review:
This movies chronicles the life and times of William Castle. He made a series of low budget horror films in the 1950s-1960s that he sold with gimmicks. In "13 Ghosts" you need viewers to see the ghost

First 20 token IDs:
[2028, 9698, 27076, 4440, 279, 2324, 323, 3115, 315, 12656, 27987, 13, 1283, 1903, 264, 4101, 315, 3428, 8199, 22169]


In [ ]:
# ============================================
# IMPROVED LSTM - BETTER PREPROCESSING
# ============================================

MAX_LENGTH = 300


def pad_sequence(tokens):

    # Truncate long reviews
    if len(tokens) > MAX_LENGTH:
        tokens = tokens[:MAX_LENGTH]

    # Pad short reviews
    else:
        tokens = tokens + [0] * (
            MAX_LENGTH - len(tokens)
        )

    return tokens


X_train_padded = [
    pad_sequence(tokens)
    for tokens in X_train_tokens
]

X_test_padded = [
    pad_sequence(tokens)
    for tokens in X_test_tokens
]


# Convert to tensors

X_train_tensor = torch.tensor(
    X_train_padded,
    dtype=torch.long
)

X_test_tensor = torch.tensor(
    X_test_padded,
    dtype=torch.long
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)


print("X_train:", X_train_tensor.shape)
print("X_test :", X_test_tensor.shape)

X_train: torch.Size([1600, 300])
X_test : torch.Size([400, 300])


In [ ]:
# ============================================
# CONVERT TO PYTORCH TENSORS
# ============================================

X_train_tensor = torch.tensor(
    X_train_padded,
    dtype=torch.long
)

X_test_tensor = torch.tensor(
    X_test_padded,
    dtype=torch.long
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)

print("Tensors created successfully!")

print("\nX_train shape:", X_train_tensor.shape)
print("X_test shape :", X_test_tensor.shape)

print("y_train shape:", y_train_tensor.shape)
print("y_test shape :", y_test_tensor.shape)

Tensors created successfully!

X_train shape: torch.Size([1600, 300])
X_test shape : torch.Size([400, 300])
y_train shape: torch.Size([1600])
y_test shape : torch.Size([400])


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("DataLoaders ready!")

DataLoaders ready!


In [ ]:
# ============================================
# IMPROVED LSTM MODEL
# ============================================

class ImprovedLSTM(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=128
    ):

        super().__init__()

        # Embedding
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        # LSTM
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )

        # Dropout
        self.dropout = nn.Dropout(0.3)

        # Output
        self.fc = nn.Linear(
            hidden_dim,
            2
        )


    def forward(self, x):

        # Token IDs → embeddings
        x = self.embedding(x)

        # LSTM
        output, (hidden, cell) = self.lstm(x)

        # Use the final hidden state
        # instead of output from padded token
        x = hidden[-1]

        # Dropout
        x = self.dropout(x)

        # Classification
        x = self.fc(x)

        return x

In [ ]:
# ============================================
# CREATE LSTM MODEL
# ============================================

vocab_size = encoding.n_vocab

model = ImprovedLSTM(
    vocab_size=vocab_size,
    embedding_dim=128,
    hidden_dim=128
)

print(model)

ImprovedLSTM(
  (embedding): Embedding(100277, 128, padding_idx=0)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)


In [ ]:
# ============================================
# LOSS AND OPTIMIZER
# ============================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0005
)

print("Model ready for training!")

Model ready for training!


In [ ]:
# ============================================
# TRAIN IMPROVED LSTM
# ============================================

epochs = 10

lstm_losses = []

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch_X, batch_y in train_loader:

        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(batch_X)

        # Loss
        loss = criterion(
            outputs,
            batch_y
        )

        # Backpropagation
        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        # Update weights
        optimizer.step()

        total_loss += loss.item()


    average_loss = (
        total_loss / len(train_loader)
    )

    lstm_losses.append(
        average_loss
    )

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/10] Loss: 0.6950
Epoch [2/10] Loss: 0.6917
Epoch [3/10] Loss: 0.6852
Epoch [4/10] Loss: 0.6572


In [ ]:
# ============================================
# EVALUATE LSTM
# ============================================

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch_X, batch_y in test_loader:

        # Get model output
        outputs = model(batch_X)

        # Get predicted class
        predictions = torch.argmax(
            outputs,
            dim=1
        )

        # Total samples
        total += batch_y.size(0)

        # Correct predictions
        correct += (
            predictions == batch_y
        ).sum().item()


lstm_accuracy = (
    100 * correct / total
)

print(
    f"LSTM Test Accuracy: "
    f"{lstm_accuracy:.2f}%"
)

In [ ]:
# ============================================
# CUSTOM REVIEW PREDICTION
# ============================================

def predict_lstm(review):

    # Tokenize
    tokens = encoding.encode(review)

    # Pad / truncate
    tokens = pad_sequence(tokens)

    # Convert to tensor
    tensor = torch.tensor(
        [tokens],
        dtype=torch.long
    )

    # Evaluation mode
    model.eval()

    with torch.no_grad():

        output = model(tensor)

        prediction = torch.argmax(
            output,
            dim=1
        ).item()

    if prediction == 1:
        return "Positive"

    else:
        return "Negative"

In [ ]:
# ============================================
# TEST NEW REVIEWS
# ============================================

reviews = [

    "This movie was absolutely fantastic. I loved every minute of it.",

    "The movie was boring and disappointing. I did not enjoy it.",

    "The acting was excellent and the story was very entertaining."

]


for review in reviews:

    prediction = predict_lstm(
        review
    )

    print("\nReview:")
    print(review)

    print("Predicted Sentiment:")
    print(prediction)

    print("-" * 50)

In [ ]:
# ============================================
# SHOW TRAINING LOSSES
# ============================================

print("LSTM Training Loss:")

for i, loss in enumerate(lstm_losses):

    print(
        f"Epoch {i + 1}: {loss:.4f}"
    )